# QueryToHint Reviewer Notebook (LLMSteer-Style)

This notebook is a reviewer-facing, end-to-end pipeline for:
- workload EDA (JOB / CEB)
- LLMSteer-style evaluation runs across embedding backbones
- merged comparison tables (JOB + CEB)
- heavy-hitter analysis for runtime anomalies

It is designed to mirror a paper workflow while remaining fully reproducible from local artifacts.

## 1. Setup

In [ ]:
import os
import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True

ROOT = Path('.').resolve()
print('ROOT:', ROOT)

## 2. Configuration

Adjust model paths, sweep preset, and seeds here.

In [ ]:
PYTHON_BIN = '/home/hashmi/miniconda/envs/query_hint/bin/python'

MODEL_PATHS = {
    'ft_mpnet': 'all_models/finetuned_mpnet_both',
    'reptile_mpnet': 'reptile_from_finetuned_all/reptile_finetuned_mpnet_both',
    'ft_minilm_l12': 'all_models/finetuned_minilm_l12_both',
    'ft_minilm_l6': 'all_models/finetuned_minilm_l6_both_v2',
    'ft_gte_small': 'all_models/finetuned_gte_small_both',
    'ft_bge_small': 'all_models/finetuned_bge_small_both',
    'ft_multiqa': 'all_models/finetuned_multiqa_minilm_l6_both',
    'ft_paraphrase': 'all_models/finetuned_paraphrase_minilm_l6_both',
    'ft_bert': 'stage1_added_models/finetuned_bert_base_both_v2',
    'ft_bge_large': 'stage1_added_models/finetuned_bge_large_both',
    'ft_bge_m3': 'stage1_added_models/finetuned_bge_m3_both',
    'ft_distilbert': 'models_finetuned/new/distilbert_sql_contrastive',
}

WORKLOADS = ['all']
SEEDS = '24508'      # expand if you want slower but broader reruns
N_SPLITS = 10
TRAIN_SIZE = 0.8

# Evaluation grid
ESTIMATORS = 'lr,svc_rbf,svc_lin,rfc,gbc'
PCS = '5,50,120'
SCALE_OPTIONS = 'false,true'
EXCLUDE_CONFIGS = 'svc_lin-pcs50-scaletrue,svc_lin-pcs120-scaletrue'

OUT_ROOT = Path('othertests/llmsteer_eval/reviewer_refresh')
OUT_ROOT.mkdir(parents=True, exist_ok=True)


## 3. Data EDA (JOB / CEB)

In [ ]:
def load_workload_csv(name):
    p = Path('data') / f'{name}.csv'
    df = pd.read_csv(
        p,
        converters={
            'hint_list': eval,
            'runtime_list': eval,
        },
    )
    df['mean_runtime'] = df['runtime_list'].apply(np.mean)
    df['sd_runtime'] = df['runtime_list'].apply(np.std)
    return df

job_df = load_workload_csv('job')
ceb_df = load_workload_csv('ceb')

print('JOB rows:', len(job_df), 'unique queries:', job_df['filename'].nunique())
print('CEB rows:', len(ceb_df), 'unique queries:', ceb_df['filename'].nunique())

job_df.head(2)

In [ ]:
eda = pd.DataFrame([
    {
        'workload': 'job',
        'rows': len(job_df),
        'unique_queries': job_df['filename'].nunique(),
        'runtime_mean': job_df['mean_runtime'].mean(),
        'runtime_p90': job_df['mean_runtime'].quantile(0.9),
        'runtime_median': job_df['mean_runtime'].median(),
    },
    {
        'workload': 'ceb',
        'rows': len(ceb_df),
        'unique_queries': ceb_df['filename'].nunique(),
        'runtime_mean': ceb_df['mean_runtime'].mean(),
        'runtime_p90': ceb_df['mean_runtime'].quantile(0.9),
        'runtime_median': ceb_df['mean_runtime'].median(),
    },
])
eda

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].hist(job_df['mean_runtime'], bins=60)
axs[0].set_title('JOB mean runtime distribution')
axs[0].set_xlabel('seconds')

axs[1].hist(ceb_df['mean_runtime'], bins=60)
axs[1].set_title('CEB mean runtime distribution')
axs[1].set_xlabel('seconds')

plt.tight_layout()
plt.show()

## 4. Run LLMSteer-Style Evaluations


In [ ]:
def run_cmd(cmd):
    print('\n>>>', ' '.join(cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f'Command failed with code {proc.returncode}')

for model_key, model_path in MODEL_PATHS.items():
    for workload in WORKLOADS:
        out_dir = OUT_ROOT / f'{model_key}_{workload}_mid'
        cmd = [
            PYTHON_BIN,
            'trainingfile/run_llmsteer_eval.py',
            '--model_path', model_path,
            '--workload', workload,
            '--seeds', SEEDS,
            '--n_splits', str(N_SPLITS),
            '--train_size', str(TRAIN_SIZE),
            '--estimators', ESTIMATORS,
            '--pcs', PCS,
            '--scale_options', SCALE_OPTIONS,
            '--exclude_configs', EXCLUDE_CONFIGS,
            '--augment_eval',
            '--output_dir', str(out_dir),
        ]
        run_cmd(cmd)


## 5. Merge Results and Build Reviewer Tables


In [ ]:
run_cmd([
    PYTHON_BIN,
    'trainingfile/merge_llmsteer_summaries.py',
    '--root', str(OUT_ROOT),
    '--output_dir', str(OUT_ROOT),
])

run_cmd([
    PYTHON_BIN,
    'trainingfile/build_table5.py',
    '--root', str(OUT_ROOT),
    '--workload', 'all',
    '--output_csv', 'table5_model_comparison.csv',
])

best_path = OUT_ROOT / 'comparison_best_by_model_workload.csv'
best_df = pd.read_csv(best_path)
table5_path = OUT_ROOT / 'table5_model_comparison.csv'
table5_df = pd.read_csv(table5_path)

print('Best configs:')
display(best_df)
print('Table 5 rows:')
display(table5_df)


In [ ]:
# Reviewer-friendly ranking by runtime objective (lower is better)
rank_df = best_df.sort_values('test_workload_sum_mean', ascending=True).copy()
rank_df[['model_path','config_name','test_workload_sum_mean','test_p90_mean','test_median_mean','f1_mean','auroc_mean']]


## 6. Heavy-Hitter Analysis (Reptile MPNet vs Contrastive MPNet)


In [ ]:
run_cmd([
    PYTHON_BIN,
    'trainingfile/analyze_heavy_hitters.py',
    '--workload', 'all',
    '--model_a_path', MODEL_PATHS['reptile_mpnet'],
    '--model_a_config', 'svc_lin-pcs50-scaleFalse',
    '--model_b_path', MODEL_PATHS['ft_mpnet'],
    '--model_b_config', 'svc_lin-pcs50-scaleFalse',
    '--seeds', '24508',
    '--n_splits', '10',
    '--train_size', '0.8',
    '--output_dir', str(OUT_ROOT / 'heavy_hitters'),
    '--prefix', 'reptile_vs_contrastive_mpnet_all',
    '--top_k', '20',
])

summary_json = OUT_ROOT / 'heavy_hitters' / 'reptile_vs_contrastive_mpnet_all_summary.json'
print(summary_json.read_text())


In [ ]:
top20_path = OUT_ROOT / 'heavy_hitters' / 'reptile_vs_contrastive_mpnet_all_top20.csv'
top20 = pd.read_csv(top20_path)
top20.head(20)


In [ ]:
plt.figure(figsize=(10,4))
plt.plot(top20['rank'], top20['cum_abs_mass_fraction'], marker='o')
plt.title('All-workload heavy-hitter concentration (Reptile MPNet vs Contrastive MPNet)')
plt.xlabel('Top-k queries by |mean delta|')
plt.ylabel('Cumulative absolute delta mass fraction')
plt.ylim(0, 1.0)
plt.show()


## 7. Export Reviewer Tables

This cell writes a compact table for paper/report inclusion.

In [ ]:
paper_cols = [
    'model_path', 'workload', 'config_name',
    'test_workload_sum_mean', 'test_workload_sum_std',
    'test_p90_mean', 'test_p90_std',
    'test_median_mean', 'test_median_std',
    'f1_mean', 'f1_std',
    'auroc_mean', 'auroc_std',
]

paper_table = best_df[paper_cols].copy()
paper_out = OUT_ROOT / 'paper_table_best_by_model.csv'
paper_table.to_csv(paper_out, index=False)
print('Saved:', paper_out)
print('Saved:', table5_path)
paper_table
